In [ ]:
# Kurulum: Colab'da repoyu indirir, yerelde hiçbir şey yapmaz.
import os, sys
REPO = "https://github.com/mehmetalibeylikci/python-cevresel-modelleme"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/python-cevresel-modelleme"):
        !git clone -q {REPO} /content/python-cevresel-modelleme
    %cd /content/python-cevresel-modelleme
    %pip install -q sweetviz
else:
    while not os.path.exists("src/model.py") and os.getcwd() != "/":
        os.chdir("..")
sys.path.insert(0, os.getcwd())

# Gün 2 — Model Kurulumu

**İYTE Çevre Topluluğu · Python ile Çevresel Modelleme**

Dün temizlediğimiz aylık yağış ve ET0 verisiyle bugün **Tahtalı Barajı'nın su bütçesi modelini** kendimiz yazacağız. Sonra modelin gerçeğe ne kadar yakın olduğunu soracağız. Cevap bizi bu eğitimin belki de en önemli dersine götürecek.

**Öğrenme hedefleri**
- Fiziksel bir sistemi **kütle korunumu** ile bir denkleme dönüştürmek
- Birimleri doğru dönüştürmek
- Modeli bir Python **fonksiyonu** olarak yazmak
- Model sonucunu gözlemlerle karşılaştırmak
- **Duyarlılık analizi** ile parametre belirsizliğinin etkisini görmek

| Süre | Konu |
|---|---|
| 0–15 | Kavramsal model: kova, kütle korunumu, birimler |
| 15–50 | Modeli fonksiyon olarak yazma |
| 50–70 | Gözlemlerle karşılaştırma: "makul mü?" |
| 70–100 | Duyarlılık analizi ve interaktif keşif |
| 100–120 | Alıştırma ve tartışma |

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

monthly = pd.read_csv("data/processed/tahtali_monthly.csv")
monthly.index = pd.to_datetime(monthly.pop("month"))
monthly.head()

> Gün 1'i tamamlamadıysanız sorun değil: `data/processed/tahtali_monthly.csv` dosyası repoda hazır olarak duruyor.

## 1. Kavramsal model: baraj bir kovadır

```
             yağış (P)
                │
  havza ──── akış (C·P·A_havza) ────►┌──────────────────┐───► buharlaşma (k·ET0·A_göl)
  546 km²                            │   GÖL  (S, hm³)  │
                                     │                  │───► taşma (dolusavaktan)
                                     └────────┬─────────┘
                                              └──────────► çekim (W) → İzmir
```

**Kütle korunumu:** Bir ayın sonunda göldeki su = ayın başındaki su + giren − çıkan.

$$S_{t+1} = S_t + \underbrace{C \cdot P \cdot A_{havza} + P \cdot A_{göl}}_{\text{giren}} - \underbrace{k_E \cdot ET_0 \cdot A_{göl}}_{\text{buharlaşma}} - \underbrace{W}_{\text{çekim}}$$

Sınırlar:
- $S$, ölü hacmin ($S_{min}$) altına inemez: oradaki su çekilemez.
- $S$, normal kottaki hacmin ($S_{maks}$) üstüne çıkamaz: fazlası **taşar**.

| Sembol | Anlamı | Değer |
|---|---|---|
| $A_{havza}$ | yağış havzası alanı | 546 km² |
| $A_{göl}$ | göl alanı | 23,52 km² |
| $S_{maks}$, $S_{min}$ | en büyük hacim / ölü hacim | 306,65 / 19,6 hm³ |
| $C$ | akış katsayısı: havzaya düşen yağmurun barajda toplanan oranı | 0,29 |
| $k_E$ | açık su buharlaşma katsayısı (FAO-56) | 1,05 |
| $W$ | aylık su çekimi | 10,7 hm³/ay |

Her değerin kaynağı `docs/parametreler.md` dosyasında. $C$ ve $W$'nin **belirsiz** olduğunu şimdiden not edin; birazdan çok önemli olacak.

### Birimler: en sık hata kaynağı

Yağış **mm**, alan **km²**, hacim **hm³** (milyon m³). Bunları nasıl birleştiririz?

$$1\ \text{mm} \times 1\ \text{km}^2 = 0{,}001\ \text{m} \times 1\,000\,000\ \text{m}^2 = 1000\ \text{m}^3 = 0{,}001\ \text{hm}^3$$

### 🟢 Alıştırma 1
Göle (23,52 km²) bir ayda 100 mm yağmur yağarsa göle doğrudan kaç hm³ su eklenir? Havzaya (546 km²) düşen 100 mm'nin $C = 0{,}29$ kadarı barajda toplanırsa bu kaç hm³ eder?

In [ ]:
MM_KM2_TO_HM3 = 0.001
# BEGIN SOLUTION
direct = 100 * 23.52 * MM_KM2_TO_HM3
runoff = 0.29 * 100 * 546 * MM_KM2_TO_HM3
# END SOLUTION
print(f"Göle doğrudan: {direct:.2f} hm³, havzadan akış: {runoff:.2f} hm³")

Havzadan gelen su, göle doğrudan düşen yağmurdan yaklaşık **7 kat** fazla. Havzanın önemi buradan geliyor.

## 2. Modeli yazalım

### 2.1 Önce tek bir ay

Simülasyona **Eylül 2019**'da başlıyoruz. Elimizdeki kaynaklı en erken gözlem, Ağustos 2019 ortasında barajın **%76** dolu olduğunu söylüyor (İZSU verisi). Bunu ağustos sonu depolaması kabul edelim.

In [ ]:
S_MAX = 306.65
S_MIN = 19.6
CATCHMENT_KM2 = 546.0
LAKE_KM2 = 23.52

s0 = S_MIN + 0.76 * (S_MAX - S_MIN)
print(f"Başlangıç depolaması: {s0:.2f} hm³")

data = monthly.loc["2019-09":]
first = data.iloc[0]
print(first)

Eylül 2019 için bir adımı elle hesaplayalım:

In [ ]:
runoff_coef, evap_coef, withdrawal = 0.29, 1.05, 10.7

inflow = (runoff_coef * first["precipitation_mm"] * CATCHMENT_KM2
          + first["precipitation_mm"] * LAKE_KM2) * MM_KM2_TO_HM3
evaporation = evap_coef * first["et0_mm"] * LAKE_KM2 * MM_KM2_TO_HM3
s1 = s0 + inflow - evaporation - withdrawal
print(f"Giren {inflow:.2f}, buharlaşma {evaporation:.2f}, çekim {withdrawal} → ay sonu {s1:.2f} hm³")

Eylül kurak bir ay: buharlaşma ve çekim, gelen suyu aştığı için baraj seviyesi düştü.

### 2.2 Bir döngü ve bir fonksiyon

Aynı adımı her ay için tekrarlamamız gerekiyor. Bu bir `for` döngüsü. Tekrar tekrar kullanabilmek için de bir **fonksiyon** içine koyalım.

### 🟢 Alıştırma 2 (birlikte yazıyoruz)
`my_simulate` fonksiyonunu tamamlayın:
1. Her ay için giren suyu ve buharlaşmayı hesaplayın.
2. Depolamayı güncelleyin.
3. Sınırları uygulayın: $S_{maks}$'ın üstündeyse taşan kısmı `spill` olarak kaydedip $S_{maks}$'a eşitleyin; $S_{min}$'in altındaysa $S_{min}$'e eşitleyin.

In [ ]:
def my_simulate(precip_mm, et0_mm, s0, runoff_coef=0.29, evap_coef=1.05, withdrawal=10.7):
    """Aylık yağış ve ET0 serilerinden ay sonu depolamalarını (hm³) hesaplar."""
    storages, spills = [], []
    storage = s0
    for precip, et0 in zip(precip_mm, et0_mm):
        spill = 0.0  # taşma yoksa 0
        # BEGIN SOLUTION
        inflow = (runoff_coef * precip * CATCHMENT_KM2 + precip * LAKE_KM2) * MM_KM2_TO_HM3
        evaporation = evap_coef * et0 * LAKE_KM2 * MM_KM2_TO_HM3
        storage = storage + inflow - evaporation - withdrawal
        if storage > S_MAX:
            spill = storage - S_MAX
            storage = S_MAX
        elif storage < S_MIN:
            storage = S_MIN
        # END SOLUTION
        storages.append(storage)
        spills.append(spill)
    return pd.DataFrame({"storage": storages, "spill": spills}, index=precip_mm.index)


result = my_simulate(data["precipitation_mm"], data["et0_mm"], s0)
result.head()

Kendi fonksiyonumuzu, repodaki test edilmiş sürümle (`src/model.py`) karşılaştıralım. Aynı sonucu veriyorsa doğru yazmışız demektir:

In [ ]:
from src.model import S0_DATE, S0_HM3, Params, active_fill_pct, simulate

reference = simulate(data["precipitation_mm"], data["et0_mm"], S0_HM3)
assert np.allclose(result["storage"], reference["storage"]), "Sonuçlar farklı; fonksiyonu kontrol edin."
print("✅ my_simulate, src/model.py ile aynı sonucu veriyor.")

`src/model.py` içindeki `simulate` biraz daha fazlasını kaydeder: her ay giren, buharlaşan, çekilen ve taşan suyu, ayrıca baraj ölü hacme indiği için **karşılanamayan çekimi** (`shortfall`). Bundan sonra onu kullanacağız.

In [ ]:
reference.head()

## 3. Model gerçeğe yakın mı?

Elimizde kaynaklı birkaç gözlem var (`data/raw/tahtali_gozlemler.csv`). Her gözlemi tarihine en yakın ay sonundaki model değeriyle karşılaştıralım.

In [ ]:
obs = pd.read_csv("data/raw/tahtali_gozlemler.csv", parse_dates=["date"])
# Ayın ilk yarısındaki gözlem bir önceki ayın sonuna, ikinci yarısındaki o ayın sonuna daha yakındır
month_start = obs["date"].dt.to_period("M").dt.to_timestamp()
obs["model_month"] = month_start.where(obs["date"].dt.day > 15, month_start - pd.DateOffset(months=1))
obs = obs[obs["model_month"] >= pd.Timestamp(S0_DATE)]  # başlangıç gözlemi hariç
obs[["date", "active_fill_pct", "model_month", "source"]]

In [ ]:
fill = active_fill_pct(reference["storage"])
obs["model_pct"] = fill.reindex(obs["model_month"]).to_numpy()
obs[["date", "active_fill_pct", "model_pct"]].round({"model_pct": 1})

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(fill.index, fill, label="Model (temel parametreler)")
ax.scatter(obs["model_month"], obs["active_fill_pct"], color="crimson", zorder=3, label="Gözlem (İZSU)")
ax.axhline(10, color="gray", linestyle="--", linewidth=1, label="Kritik seviye (%10)")
ax.set_ylim(0, 105)
ax.set_title("Tahtalı aktif doluluk: model ve gözlem")
ax.set_ylabel("Aktif doluluk (%)")
ax.legend(loc="lower left")
plt.show()

### 🤔 Durup düşünelim

Model, 2022–2025 kuraklığının **derinliğini yakalayamıyor.** Modelde doluluk hiç %40'ın altına inmiyor; gerçekte ise Ağustos 2025'te %7,45'e kadar düştü.

Bu, modeli çöpe atmamız gerektiği anlamına mı geliyor? Henüz değil. Önce şunu soralım: **Parametrelerimizden ne kadar eminiz?**
- $C = 0{,}29$ değerini kaba bir hesapla türettik.
- $W = 10{,}7$ hm³/ay bir **tasarım değeri**; bazı haber kaynaklarında yıllık 205 hm³ (17,1 hm³/ay) da geçiyor.

Bu belirsizliklerin sonuca etkisini ölçmenin yolu **duyarlılık analizidir.**

## 4. Duyarlılık analizi

Parametreleri bir aralıkta tarayıp her kombinasyon için modeli yeniden çalıştırıyoruz. `Params` nesnesinin bir kopyasını, istediğimiz alanı değiştirerek `dataclasses.replace` ile üretiyoruz:

In [ ]:
from dataclasses import replace

base = Params()
print(base)
print(replace(base, runoff_coef=0.2))

In [ ]:
runoff_values = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
withdrawal_values = [8.0, 10.7, 13.0, 17.1]

rows = []
for c in runoff_values:
    for w in withdrawal_values:
        out = simulate(data["precipitation_mm"], data["et0_mm"], S0_HM3, replace(base, runoff_coef=c, withdrawal=w))
        rows.append({
            "runoff_coef": c,
            "withdrawal": w,
            "fill_2026_09": active_fill_pct(out["storage"].loc["2026-09-01"]),
            "months_below_10": int((active_fill_pct(out["storage"]) < 10).sum()),
        })
sweep = pd.DataFrame(rows)
sweep.pivot(index="runoff_coef", columns="withdrawal", values="fill_2026_09").round(0)

In [ ]:
import seaborn as sns

table = sweep.pivot(index="runoff_coef", columns="withdrawal", values="fill_2026_09")
fig, ax = plt.subplots(figsize=(6, 4.5))
sns.heatmap(table, annot=True, fmt=".0f", cmap="RdYlBu", vmin=0, vmax=100,
            cbar_kws={"label": "Eylül 2026 aktif doluluk (%)"}, ax=ax)
ax.set_xlabel("Çekim W (hm³/ay)")
ax.set_ylabel("Akış katsayısı C")
ax.set_title("Duyarlılık: Eylül 2026'da baraj ne kadar dolu?\n(gözlem: Ekim 2026 ortası %42,75)")
plt.show()

Bu tabloyu okuyalım:
- **Sonuç, parametrelere son derece hassas.** $C$'deki 0,05'lik bir değişiklik bile barajı "boş" ile "dolu" arasında gidip getirebiliyor.
- Gözlemle (%42,75) uyumlu kombinasyonlar, haritada **köşegen boyunca dar bir şerit** oluşturuyor. Daha yüksek $C$ ile daha yüksek $W$ birbirini dengeliyor.
- Yani elimizdeki birkaç gözlem $C$ ile $W$'yi **ayrı ayrı belirlemeye yetmiyor.** Buna modelcilikte *eşsonuçluluk* (equifinality) denir.

### 🔵 Alıştırma 3
Her kombinasyon için modelin **tüm gözlemlerle** ne kadar uyuştuğunu ölçün. Kök ortalama kare hata (RMSE) hesaplayın:

$$RMSE = \sqrt{\frac{1}{n}\sum (\text{model} - \text{gözlem})^2}$$

`sweep` tablosuna `rmse` sütunu ekleyin ve RMSE'si en küçük 5 kombinasyonu gösterin. Bu kombinasyonlar birbirine benziyor mu?

In [ ]:
# BEGIN SOLUTION
def rmse_for(c, w):
    out = simulate(data["precipitation_mm"], data["et0_mm"], S0_HM3, replace(base, runoff_coef=c, withdrawal=w))
    model_pct = active_fill_pct(out["storage"]).reindex(obs["model_month"]).to_numpy()
    return float(np.sqrt(np.mean((model_pct - obs["active_fill_pct"].to_numpy()) ** 2)))


sweep["rmse"] = [rmse_for(c, w) for c, w in zip(sweep["runoff_coef"], sweep["withdrawal"])]
print(sweep.nsmallest(5, "rmse").round(1))
# END SOLUTION

> ⚠️ En küçük RMSE'yi veren kombinasyonu seçip "doğru parametre bu" demek **kalibrasyondur.** Bu eğitimde bunu bilerek yapmıyoruz: yalnızca 4 gözlemimiz var (başlangıç değeri hariç) ve bu gözlemler farklı kaynaklardan geliyor. 4 noktaya 2 parametre uydurmak, modelin gerçeği açıkladığını değil, eğrinin noktalara esneyebildiğini gösterir.

### İnteraktif keşif

Aşağıdaki kaydırma çubuklarıyla parametreleri değiştirin ve grafiğin nasıl değiştiğini izleyin. (Colab'da çubuklar hücre çalıştırılınca görünür.)

In [ ]:
from ipywidgets import FloatSlider, interact


def explore(runoff_coef=0.29, withdrawal=10.7):
    out = simulate(data["precipitation_mm"], data["et0_mm"], S0_HM3,
                   replace(base, runoff_coef=runoff_coef, withdrawal=withdrawal))
    pct = active_fill_pct(out["storage"])
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.plot(pct.index, pct, label="Model")
    ax.scatter(obs["model_month"], obs["active_fill_pct"], color="crimson", zorder=3, label="Gözlem")
    ax.axhline(10, color="gray", linestyle="--", linewidth=1)
    ax.set_ylim(0, 105)
    ax.set_ylabel("Aktif doluluk (%)")
    ax.set_title(f"C = {runoff_coef:.2f}, W = {withdrawal:.1f} hm³/ay")
    ax.legend(loc="lower left")
    plt.show()


interact(
    explore,
    runoff_coef=FloatSlider(value=0.29, min=0.05, max=0.5, step=0.01, description="C"),
    withdrawal=FloatSlider(value=10.7, min=5, max=20, step=0.1, description="W (hm³/ay)"),
);

## 5. Tartışma: modelimizin varsayımları

Model gerçeği tam yakalayamıyor. Olası nedenleri tartışalım:

1. **Tek nokta yağışı:** ERA5'in ~25 km'lik tek bir hücresi, 546 km²'lik havzanın tamamını temsil ediyor mu?
2. **Sabit akış katsayısı:** Kuru toprak yağmuru emer, ıslak toprak emmez. $C$ gerçekte mevsime ve önceki aylara bağlı.
3. **Sabit çekim:** İZSU kuraklıkta çekimi azaltır, nüfus arttıkça talep artar.
4. **Sabit göl alanı:** Baraj boşaldıkça göl küçülür, buharlaşma azalır. (Aşağıdaki 🔵 alıştırma)
5. **Eksik süreçler:** sızma, yeraltı suyu, kaçak kullanım.

Bu eksikleri bilmek modeli değersiz kılmaz. Aksine, modelin **neye cevap verebileceğini** belirler. Yarın bu modeli mutlak tahmin için değil, **senaryoları karşılaştırmak** için kullanacağız.

🎉 **Gün 2 tamam!** Yazdığınız fonksiyonun test edilmiş hâli `src/model.py` dosyasında duruyor; Gün 3'te onu kullanacağız.

---
## Zaman kalırsa

### 🔵 Alıştırma 4: Depolamaya bağlı göl alanı
Gerçekte göl alanı depolamayla birlikte değişir. Basit bir yaklaşım:

$$A_{göl}(S) = A_{göl,maks} \cdot \left(\frac{S}{S_{maks}}\right)^{2/3}$$

(Hacim üç boyutlu, alan iki boyutlu olduğu için 2/3 kuvveti.)

`my_simulate` fonksiyonunu kopyalayıp `my_simulate_area` adıyla bu formülü kullanacak şekilde değiştirin. Göl alanını her ay, **ayın başındaki** depolamaya göre hesaplayın. Temel parametrelerle iki modeli aynı grafikte karşılaştırın. Fark büyük mü?

In [ ]:
# BEGIN SOLUTION
def my_simulate_area(precip_mm, et0_mm, s0, runoff_coef=0.29, evap_coef=1.05, withdrawal=10.7):
    storages = []
    storage = s0
    for precip, et0 in zip(precip_mm, et0_mm):
        lake_km2 = LAKE_KM2 * (storage / S_MAX) ** (2 / 3)
        inflow = (runoff_coef * precip * CATCHMENT_KM2 + precip * lake_km2) * MM_KM2_TO_HM3
        evaporation = evap_coef * et0 * lake_km2 * MM_KM2_TO_HM3
        storage = min(max(storage + inflow - evaporation - withdrawal, S_MIN), S_MAX)
        storages.append(storage)
    return pd.Series(storages, index=precip_mm.index)


area_model = my_simulate_area(data["precipitation_mm"], data["et0_mm"], s0)
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(reference.index, reference["storage"], label="Sabit göl alanı")
ax.plot(area_model.index, area_model, label="Depolamaya bağlı göl alanı")
ax.set_ylabel("Depolama (hm³)")
ax.legend()
plt.show()
# END SOLUTION